In [1]:
%%writefile ../src/api/schemas.py
from pydantic import BaseModel
from typing import List

class RecommendRequest(BaseModel):
    n: float
    p: float
    k: float
    temperature: float
    humidity: float
    ph: float
    rainfall: float

class FeatureContribution(BaseModel):
    feature: str
    shap_value: float

class RecommendResponse(BaseModel):
    predicted_crop: str
    confidence: float
    top_features: List[FeatureContribution]
    explanation_text: str

class YieldRequest(BaseModel):
    state_name: str
    crop: str
    season: str
    area_ha: float
    latitude: float
    longitude: float

class YieldResponse(BaseModel):
    predicted_yield_kg_per_ha: float
    top_features: List[FeatureContribution]
    explanation_text: str
    weather_used: dict

Writing ../src/api/schemas.py


In [2]:
%%writefile ../src/api/model_loader.py
"""
Loads all models, scalers, encoders, and SHAP explainers once at API startup.
"""
import tensorflow as tf
import joblib
import shap
import numpy as np
import pandas as pd
import os

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

class Models:
    def __init__(self):
        self.model_a = tf.keras.models.load_model(os.path.join(BASE_DIR, "models/saved/model_a_crop_recommendation.keras"))
        self.scaler_a = joblib.load(os.path.join(BASE_DIR, "models/saved/scaler_a.pkl"))
        self.le_a = joblib.load(os.path.join(BASE_DIR, "models/saved/label_encoder_a.pkl"))

        df_a = pd.read_csv(os.path.join(BASE_DIR, "../data/processed/dataset3_clean.csv"))
        X_a = df_a.drop(columns=['crop'])
        X_a_scaled = self.scaler_a.transform(X_a)
        background_a = X_a_scaled[np.random.choice(X_a_scaled.shape[0], 100, replace=False)]
        self.explainer_a = shap.KernelExplainer(self.model_a.predict, background_a)

        self.model_b = tf.keras.models.load_model(os.path.join(BASE_DIR, "models/saved/model_b_yield_prediction.keras"))
        self.scaler_b = joblib.load(os.path.join(BASE_DIR, "models/saved/scaler_b.pkl"))
        self.model_b_cols = joblib.load(os.path.join(BASE_DIR, "models/saved/model_b_columns.pkl"))

        df_b = pd.read_csv(os.path.join(BASE_DIR, "../data/processed/model_b_training_data.csv"))
        df_b_encoded = pd.get_dummies(df_b, columns=['state_name', 'crop', 'season'], drop_first=True)
        df_b_encoded = df_b_encoded.reindex(columns=self.model_b_cols, fill_value=0)
        X_b_scaled_full = self.scaler_b.transform(df_b_encoded.astype(float))
        background_b = X_b_scaled_full[np.random.choice(X_b_scaled_full.shape[0], 100, replace=False)]
        self.explainer_b = shap.KernelExplainer(self.model_b.predict, background_b)

        print("All models and explainers loaded")

models = Models()


Writing ../src/api/model_loader.py


In [3]:
%%writefile ../src/api/app.py
"""
AgriVision AI - FastAPI backend
Exposes crop recommendation and yield prediction with SHAP explainability.
"""
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

from api.schemas import RecommendRequest, RecommendResponse, YieldRequest, YieldResponse
from api.model_loader import models
from explainability.shap_utils import explain_crop_recommendation, explain_yield_prediction
from services.weather_service import get_live_weather

app = FastAPI(title="AgriVision AI API")

# allow the frontend (running on a different port/domain during dev) to call this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # tighten this to your actual frontend URL before production
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def root():
    return {"status": "AgriVision AI API is running"}

@app.post("/recommend", response_model=RecommendResponse)
def recommend_crop(req: RecommendRequest):
    try:
        result = explain_crop_recommendation(
            models.model_a, models.scaler_a, models.le_a, models.explainer_a,
            n=req.n, p=req.p, k=req.k, temperature=req.temperature,
            humidity=req.humidity, ph=req.ph, rainfall=req.rainfall
        )
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/predict-yield", response_model=YieldResponse)
def predict_yield(req: YieldRequest):
    try:
        weather = get_live_weather(req.latitude, req.longitude)
        result = explain_yield_prediction(
            models.model_b, models.scaler_b, models.model_b_cols, models.explainer_b,
            state_name=req.state_name, crop=req.crop, season=req.season, area_ha=req.area_ha,
            temperature_c=weather['temperature_c'], humidity_pct=weather['humidity_pct'],
            rainfall_mm=weather['rainfall_mm'], wind_speed_m_s=weather['wind_speed_m_s'],
            solar_radiation_mj_m2_day=weather['solar_radiation_mj_m2_day']
        )
        result['weather_used'] = weather
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

Writing ../src/api/app.py


In [ ]:
%%writefile ../src/explainability/shap_utils.py
"""
SHAP-based explanation functions for AgriVision AI.
Model A: crop recommendation (classification)
Model B: yield prediction (regression)
"""
import numpy as np

def explain_crop_recommendation(model_a, scaler_a, le_a, explainer_a, 
                                  n, p, k, temperature, humidity, ph, rainfall, top_n=3):
    input_arr = np.array([[n, p, k, temperature, humidity, ph, rainfall]])
    input_scaled = scaler_a.transform(input_arr)
    
    pred_probs = model_a.predict(input_scaled, verbose=0)
    pred_class_idx = np.argmax(pred_probs[0])
    pred_crop = le_a.classes_[pred_class_idx]
    confidence = pred_probs[0][pred_class_idx]
    
    shap_vals = explainer_a.shap_values(input_scaled, nsamples=100)
    crop_shap = shap_vals[0, :, pred_class_idx]
    
    feature_names = ['N','P','K','temperature','humidity','ph','rainfall']
    contributions = list(zip(feature_names, crop_shap))
    contributions.sort(key=lambda x: abs(x[1]), reverse=True)
    top_features = contributions[:top_n]
    
    explanation_parts = []
    for feat, val in top_features:
        direction = "increased" if val > 0 else "decreased"
        explanation_parts.append(f"{feat} {direction} confidence")
    
    explanation_text = f"Recommended: {pred_crop} ({confidence:.1%} confidence). Key factors: " + ", ".join(explanation_parts) + "."
    
    return {
        "predicted_crop": pred_crop,
        "confidence": float(confidence),
        "top_features": [{"feature": f, "shap_value": float(v)} for f, v in top_features],
        "explanation_text": explanation_text
    }


def explain_yield_prediction(model_b, scaler_b, model_b_cols, explainer_b,
                               state_name, crop, season, area_ha, temperature_c, humidity_pct,
                               rainfall_mm, wind_speed_m_s, solar_radiation_mj_m2_day, top_n=3):
    input_dict = {col: 0 for col in model_b_cols}
    input_dict['area_ha'] = area_ha
    input_dict['temperature_c'] = temperature_c
    input_dict['humidity_pct'] = humidity_pct
    input_dict['rainfall_mm'] = rainfall_mm
    input_dict['wind_speed_m_s'] = wind_speed_m_s
    input_dict['solar_radiation_mj_m2_day'] = solar_radiation_mj_m2_day
    
    state_col = f'state_name_{state_name}'
    crop_col = f'crop_{crop}'
    season_col = f'season_{season}'
    if state_col in input_dict: input_dict[state_col] = 1
    if crop_col in input_dict: input_dict[crop_col] = 1
    if season_col in input_dict: input_dict[season_col] = 1
    
    input_arr = np.array([[input_dict[col] for col in model_b_cols]])
    input_scaled = scaler_b.transform(input_arr)
    
    pred_yield = model_b.predict(input_scaled, verbose=0)[0][0]
    
    shap_vals = explainer_b.shap_values(input_scaled, nsamples=100)
    feature_shap = shap_vals[0, :, 0]
    
    contributions = list(zip(model_b_cols, feature_shap))
    contributions.sort(key=lambda x: abs(x[1]), reverse=True)
    top_features = contributions[:top_n]
    
    explanation_parts = []
    for feat, val in top_features:
        direction = "increased" if val > 0 else "decreased"
        clean_name = feat.replace('state_name_','').replace('crop_','').replace('season_','')
        is_active = input_dict[feat] == 1
        
        if feat.startswith('state_name_'):
            phrase = f"being in {clean_name}" if is_active else f"not being in {clean_name}"
        elif feat.startswith('crop_'):
            phrase = f"the crop being {clean_name}" if is_active else f"the crop not being {clean_name}"
        elif feat.startswith('season_'):
            phrase = f"the season being {clean_name}" if is_active else f"the season not being {clean_name}"
        else:
            phrase = clean_name
        
        explanation_parts.append(f"{phrase} {direction} predicted yield")
    
    explanation_text = f"Predicted yield: {pred_yield:.0f} kg/ha. Key factors: " + ", ".join(explanation_parts) + "."
    
    return {
        "predicted_yield_kg_per_ha": float(pred_yield),
        "top_features": [{"feature": f, "shap_value": float(v)} for f, v in top_features],
        "explanation_text": explanation_text
    }

In [1]:
with open("../src/api/app.py") as f:
    print(f.read())

"""
AgriVision AI - FastAPI backend
Exposes crop recommendation and yield prediction with SHAP explainability.
"""
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

from api.schemas import RecommendRequest, RecommendResponse, YieldRequest, YieldResponse
from api.model_loader import models
from explainability.shap_utils import explain_crop_recommendation, explain_yield_prediction
from services.weather_service import get_live_weather

app = FastAPI(title="AgriVision AI API")

# allow the frontend (running on a different port/domain during dev) to call this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # tighten this to your actual frontend URL before production
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def root():
    return {"status": "AgriVision AI API is running"}

@app.post("/recommend", respon

In [2]:
%%writefile ../src/api/app.py
"""
AgriVision AI - FastAPI backend
Exposes crop recommendation and yield prediction with SHAP explainability.
"""
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

from api.schemas import RecommendRequest, RecommendResponse, YieldRequest, YieldResponse
from api.model_loader import models
from explainability.shap_utils import explain_crop_recommendation, explain_yield_prediction
from services.weather_service import get_live_weather

app = FastAPI(title="AgriVision AI API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

VALID_CROPS_YIELD = ['rice', 'maize', 'chickpea', 'cotton']
VALID_SEASONS = ['Kharif', 'Rabi']

@app.get("/")
def root():
    return {"status": "AgriVision AI API is running"}

@app.post("/recommend", response_model=RecommendResponse)
def recommend_crop(req: RecommendRequest):
    if not (0 <= req.ph <= 14):
        raise HTTPException(status_code=400, detail="pH must be between 0 and 14")
    if req.humidity < 0 or req.humidity > 100:
        raise HTTPException(status_code=400, detail="Humidity must be between 0 and 100")
    if req.n < 0 or req.p < 0 or req.k < 0 or req.rainfall < 0:
        raise HTTPException(status_code=400, detail="N, P, K, and rainfall must be non-negative")

    try:
        result = explain_crop_recommendation(
            models.model_a, models.scaler_a, models.le_a, models.explainer_a,
            n=req.n, p=req.p, k=req.k, temperature=req.temperature,
            humidity=req.humidity, ph=req.ph, rainfall=req.rainfall
        )
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction failed: {str(e)}")

@app.post("/predict-yield", response_model=YieldResponse)
def predict_yield(req: YieldRequest):
    if req.crop.lower() not in VALID_CROPS_YIELD:
        raise HTTPException(
            status_code=400,
            detail=f"Yield prediction is only available for: {', '.join(VALID_CROPS_YIELD)}"
        )
    if req.season not in VALID_SEASONS:
        raise HTTPException(
            status_code=400,
            detail=f"Season must be one of: {', '.join(VALID_SEASONS)}"
        )
    if req.area_ha <= 0:
        raise HTTPException(status_code=400, detail="Area must be greater than 0")
    if not (-90 <= req.latitude <= 90) or not (-180 <= req.longitude <= 180):
        raise HTTPException(status_code=400, detail="Invalid latitude/longitude")

    try:
        weather = get_live_weather(req.latitude, req.longitude)
    except Exception as e:
        raise HTTPException(status_code=502, detail=f"Failed to fetch live weather data: {str(e)}")

    try:
        result = explain_yield_prediction(
            models.model_b, models.scaler_b, models.model_b_cols, models.explainer_b,
            state_name=req.state_name, crop=req.crop, season=req.season, area_ha=req.area_ha,
            temperature_c=weather['temperature_c'], humidity_pct=weather['humidity_pct'],
            rainfall_mm=weather['rainfall_mm'], wind_speed_m_s=weather['wind_speed_m_s'],
            solar_radiation_mj_m2_day=weather['solar_radiation_mj_m2_day']
        )
        result['weather_used'] = weather
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction failed: {str(e)}")

Overwriting ../src/api/app.py


In [3]:
with open("../src/api/app.py") as f:
    print(f.read())

"""
AgriVision AI - FastAPI backend
Exposes crop recommendation and yield prediction with SHAP explainability.
"""
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

from api.schemas import RecommendRequest, RecommendResponse, YieldRequest, YieldResponse
from api.model_loader import models
from explainability.shap_utils import explain_crop_recommendation, explain_yield_prediction
from services.weather_service import get_live_weather

app = FastAPI(title="AgriVision AI API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

VALID_CROPS_YIELD = ['rice', 'maize', 'chickpea', 'cotton']
VALID_SEASONS = ['Kharif', 'Rabi']

@app.get("/")
def root():
    return {"status": "AgriVision AI API is running"}

@app.post("/recommend", response_model=RecommendResponse)
def recommend_crop(req: 

In [4]:
%%writefile ../../docs/api_contract.md
# AgriVision AI - API Contract

Base URL (local development): `http://127.0.0.1:8000`

## 1. Health Check

**GET** `/`

Response:
```json
{"status": "AgriVision AI API is running"}
```

## 2. Crop Recommendation

**POST** `/recommend`

Request body:
```json
{
  "n": 90,
  "p": 42,
  "k": 43,
  "temperature": 20.9,
  "humidity": 82.0,
  "ph": 6.5,
  "rainfall": 202.9
}
```

| Field | Type | Notes |
|---|---|---|
| n, p, k | float | Soil nutrient levels (nitrogen, phosphorus, potassium), non-negative |
| temperature | float | Degrees Celsius |
| humidity | float | Percentage, 0-100 |
| ph | float | Soil pH, 0-14 |
| rainfall | float | mm |

Success response (200):
```json
{
  "predicted_crop": "rice",
  "confidence": 0.7222487926483154,
  "top_features": [
    {"feature": "rainfall", "shap_value": 0.31},
    {"feature": "humidity", "shap_value": 0.10}
  ],
  "explanation_text": "Recommended: rice (72.2% confidence). Key factors: rainfall increased confidence, humidity increased confidence."
}
```

Can recommend any of 22 crops: apple, banana, blackgram, chickpea, coconut, coffee, cotton, grapes, jute, kidneybeans, lentil, maize, mango, mothbeans, mungbean, muskmelon, orange, papaya, pigeonpeas, pomegranate, rice, watermelon.

Error response (400): invalid pH/humidity/negative values.

## 3. Yield Prediction

**POST** `/predict-yield`

Request body:
```json
{
  "state_name": "Chhattisgarh",
  "crop": "rice",
  "season": "Kharif",
  "area_ha": 10000,
  "latitude": 21.1982964,
  "longitude": 81.4007922
}
```

| Field | Type | Notes |
|---|---|---|
| state_name | string | Indian state name |
| crop | string | **Must be one of: rice, maize, chickpea, cotton** |
| season | string | **Must be Kharif or Rabi** |
| area_ha | float | Farm area in hectares, must be > 0 |
| latitude, longitude | float | Used to fetch live weather via NASA POWER |

Weather is fetched automatically server-side (most recent complete year) — the frontend does not need to supply weather data.

Success response (200):
```json
{
  "predicted_yield_kg_per_ha": 1467.6,
  "top_features": [
    {"feature": "state_name_Chhattisgarh", "shap_value": -649.65},
    {"feature": "crop_rice", "shap_value": 410.14}
  ],
  "explanation_text": "Predicted yield: 1468 kg/ha. Key factors: being in Chhattisgarh decreased predicted yield, the crop being rice increased predicted yield.",
  "weather_used": {
    "temperature_c": 25.36,
    "humidity_pct": 64.46,
    "rainfall_mm": 1609.65,
    "wind_speed_m_s": 1.94,
    "solar_radiation_mj_m2_day": 17.32,
    "data_year": 2025
  }
}
```

Error responses:
- 400: invalid crop (not one of the 4 supported), invalid season, area ≤ 0, invalid lat/long
- 502: NASA POWER weather fetch failed
- 500: internal prediction error

## Frontend Notes

- `top_features` is sorted by impact (most influential first) — recommend showing this as a horizontal bar chart, with positive SHAP values in one color and negative in another.
- `explanation_text` is ready-to-display plain English — no need to construct sentences from `top_features` yourself, though you can use both (chart from `top_features`, caption from `explanation_text`).
- Yield prediction only supports 4 crops currently (rice, maize, chickpea, cotton) — the recommendation endpoint supports all 22. Consider disabling/filtering the yield form to these 4 crops, or gracefully handling the 400 error if a user picks something else.
- CORS is open (`*`) in development. This will be tightened before deployment — confirm your dev server's origin/port with the backend team if you hit CORS issues later.

Writing ../../docs/api_contract.md


In [5]:
with open("../../docs/api_contract.md") as f:
    print(f.read()[:500])

# AgriVision AI - API Contract

Base URL (local development): `http://127.0.0.1:8000`

## 1. Health Check

**GET** `/`

Response:
```json
{"status": "AgriVision AI API is running"}
```

## 2. Crop Recommendation

**POST** `/recommend`

Request body:
```json
{
  "n": 90,
  "p": 42,
  "k": 43,
  "temperature": 20.9,
  "humidity": 82.0,
  "ph": 6.5,
  "rainfall": 202.9
}
```

| Field | Type | Notes |
|---|---|---|
| n, p, k | float | Soil nutrient levels (nitrogen, phosphorus, potassium), non-negat
